# Phase 2 — ColBERT contrastive fine-tune (R8 stage 1-3)

Reuses the probe's PLAID setup. Pipeline: build train triples (`colbert_data`) -> fine-tune with PyLate `Contrastive` + turn-1 dev-recall selection (`colbert_finetune`, the ORIGINAL feedback way) -> full-catalog PLAID gate (`colbert_index`) measuring recall@k vs the fused pool (G3).

Train query == serve query via `QueryBuilder(recency_window=1)` (focused: last utterance + goal). Base = GTE-ModernColBERT. Spec: `.claude/documents/features/Colbert_Improved.md`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)

## 2. Clone + install (pylate)

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate
import sys; sys.path.insert(0,'.')

## 3. Config

In [ ]:
COLBERT_MODEL='lightonai/GTE-ModernColBERT-v1'   # base for the fine-tune (stronger than colbertv2.0)
Q_LEN=96; D_LEN=300; BSIZE=128; MAXK=500; KS=[50,100,200,500]
EXPANSION_FIRST=True            # keep doc2query text under doc_maxlen truncation (R8 §4.3)
# --- fine-tune knobs ---
TRAIN_SESSIONS=0               # 0 = all 15k train sessions; set small (e.g. 2000) for a smoke run
SEL_SESSIONS=500             # test sessions [0:SEL] drive checkpoint SELECTION (dev-eval callback)
GATE_SESSIONS=1000           # test sessions [SEL:GATE] are the HELD-OUT G3 gate (selection never sees them)
POOL_SIZE=100                 # fused-pool depth for hard-neg mining + dev pack
K_NEGS=15                     # hard negatives per positive (the original way)
EPOCHS=2.0; LR=1e-5; TRAIN_BS=32; EVAL_STEPS=500; DEV_SUBSET=300; SEL_K=20; SEED=42
ORG='talkpl-ai'
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'
TRAIN_JSONL=f'{OUT}/colbert_train.jsonl'
OUT_DIR=f'{OUT}/colbert/music-colbert-v1'                 # fine-tuned checkpoint (best by dev recall)
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_NAME='music-colbert-v1-d%d'%D_LEN
print('base', COLBERT_MODEL, '| out', OUT_DIR)

## 4. Load enriched catalog + TRAIN + DEV + fusion (hard-neg pool == serve pool)

In [ ]:
import glob, os, pickle
import pandas as pd
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion
from mcrs.retrieval.colbert_channel import colbert_doc_text

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB)); assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)}')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names); CKNN_MODS={l:m for l,m in CONTENT_MODALITIES.items() if m in _avail}
te={l:TrackEmbeddings(tre.select_columns(['track_id',m]),modalities=[m]) for l,m in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
raw_train = dsd['train'] if not TRAIN_SESSIONS else dsd['train'].select(range(TRAIN_SESSIONS))
conv_tr=Conversations(raw_train)
conv_sel =Conversations(dsd['test'].select(range(0, SEL_SESSIONS)))            # checkpoint selection
conv_gate=Conversations(dsd['test'].select(range(SEL_SESSIONS, GATE_SESSIONS)))  # HELD-OUT G3 gate

model=SentenceTransformer(DENSE_MODEL,device='cuda')
doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
cooc=pickle.load(open(COOC_PKL,'rb')) if os.path.exists(COOC_PKL) else build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat))
if not os.path.exists(COOC_PKL): pickle.dump(cooc,open(COOC_PKL,'wb'))
cknn=[ContentKNNChannel(te[l],m,label=l) for l,m in CKNN_MODS.items()]
chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn, CFChannel(ue,te_cf,'cf-bpr'),
       SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
fusion=RRFFusion(chans,k=60); print('channels:',[c.label for c in chans])

qb=QueryBuilder(recency_window=1)                       # focused query: last utterance + goal (train==serve)
doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=EXPANSION_FIRST)

## 5. Build train triples (TRAIN split) — contrastive (query, +, hard-negs)

In [ ]:
from mcrs.training.colbert_data import build_colbert_train_data, write_jsonl
rep={}
triples=build_colbert_train_data(conv_tr, raw_train, qb, fusion, doc_fn,
                                 pool_size=POOL_SIZE, k_negs=K_NEGS, show_progress=True, report=rep)
n=write_jsonl(triples, TRAIN_JSONL)
print('triples:', n, '| report:', rep, '->', TRAIN_JSONL)

## 6. Dev-eval pack (TEST split, turn-1 gate) for checkpoint selection

In [ ]:
from mcrs.training.colbert_data import build_dev_eval_pack
dev_pack=build_dev_eval_pack(conv_sel, qb, fusion, doc_fn, pool_size=POOL_SIZE)
print('dev pack: turn-1 queries', len(dev_pack['queries']), '| pool docs', len(dev_pack['tid_to_text']))

## 7. Fine-tune (PyLate Contrastive) + dev-recall selection — the original feedback

In [ ]:
from mcrs.training.colbert_finetune import train_colbert
train_colbert(triples, OUT_DIR, base_model=COLBERT_MODEL, dev_eval_pack=dev_pack,
              epochs=EPOCHS, batch_size=TRAIN_BS, lr=LR, eval_steps=EVAL_STEPS,
              dev_subset=DEV_SUBSET, q_len=Q_LEN, d_len=D_LEN, k=SEL_K, seed=SEED, force=False)
print('best checkpoint ->', OUT_DIR)

## 8. G3 gate — full-catalog PLAID recall of the fine-tuned model vs the fused pool

In [ ]:
from pylate import models
from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
from mcrs.data.ids import canonical_track_id
from mcrs.eval.probe import recall_ceiling

ft=models.ColBERT(model_name_or_path=OUT_DIR, query_length=Q_LEN, document_length=D_LEN)
retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, FT_IDX_NAME, doc_fn, batch_size=BSIZE)

dv=list(conv_gate.turns())   # held-out: NOT used for selection
cb_queries=[qb.build(t).text for t in dv]              # SAME focused query as train
colbert_lists=colbert_retrieve(ft, retr, cb_queries, MAXK)

queries=[qb.build(t).text for t in dv]
bc=[{'history_tids':t.history_tids,'user_id':t.user_id} for t in dv]; uids=[t.user_id for t in dv]
per_channel={ch.label: ch.batch_text_to_item_retrieval(queries, MAXK, bc, uids) for ch in fusion.channels}
per_channel['colbert_ft']=colbert_lists
golds=[conv_gate.gold(t.session_id,t.turn_number) for t in dv]; segments=[t.segment for t in dv]
rep_all =recall_ceiling(per_channel, golds, KS, segments)
rep_base=recall_ceiling({k:v for k,v in per_channel.items() if k!='colbert_ft'}, golds, KS, segments)

## 9. Read the gate (G3)

In [ ]:
def r(d): return {k: round(d[k],4) for k in KS}
cb=rep_all['per_channel']['colbert_ft']; dn=rep_all['per_channel']['dense']
print('colbert_ft recall :', r(cb['recall']), '| unique:', round(cb['unique_recall'],4))
print('  cold            :', r(cb['by_segment']['cold']))
print('  warm            :', r(cb['by_segment']['warm']))
print('dense      recall :', r(dn['recall']), '| unique:', round(dn['unique_recall'],4))
print('FUSED w/o ft      :', r(rep_base['fused']['recall']))
print('FUSED w/  ft      :', r(rep_all['fused']['recall']))
lift={k: round(rep_all['fused']['recall'][k]-rep_base['fused']['recall'][k],4) for k in KS}
print('FUSED recall LIFT :', lift)
# G3: beats dense @200 OR fused lift>=+0.01 @200 OR unique>=0.02
g3 = cb['recall'][200] > dn['recall'][200] or lift[200] >= 0.01 or cb['unique_recall'] >= 0.02
print('\nG3 GATE:', 'PASS — keep + wire into R7 / re-probe' if g3 else 'FAIL — keep cold-only or drop (recall wall)')